In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import sys

In [ ]:
root_dir = Path().resolve().parents[0]
data_dir = root_dir / "data" / "rmp_ucsd_reviews.csv"
sys.path.append(str(root_dir))
df = pd.read_csv(data_dir)


In [ ]:
COLUMNS_TO_KEEP = [
    "date",
    "class",
    "qualityRating",
    "difficultyRatingRounded",
    "grade",
    "comment",
]
df = df[COLUMNS_TO_KEEP].copy()
valid_course_pattern = r"^[A-Z]{3,}[0-9]+[A-Z]?$"
df = df[df["class"].str.match(valid_course_pattern, na=False)].copy()
df

In [ ]:
df = df.dropna().copy()
df = df.rename(columns={
    "qualityRating": "quality",
    "clarityRatingRounded": "clarity",
    "difficultyRatingRounded": "difficulty",
})
df

In [ ]:
df_clean = df.copy()
df_clean["date"] = df_clean["date"].str.replace(r'\s+UTC$', '', regex=True)
df_clean["date"] = pd.to_datetime(df_clean["date"], utc=True).dt.tz_localize(None)

course = "PHYS2A"
df_course = df_clean[df_clean["class"].str.upper() == course].copy()
df_course = df_course[
    (df_course["date"] >= pd.Timestamp("2016-01-01")) &
    (df_course["date"] <= pd.Timestamp("2026-12-31"))
].sort_values("date").copy()

plt.figure(figsize=(10,5))
plt.plot(df_course["date"], df_course["quality"], marker="o", label="quality")
plt.plot(df_course["date"], df_course["difficulty"], marker="o", label="difficulty")
plt.ylim(1,5)
plt.xlabel("Date")
plt.ylabel("Rating Score")
plt.title("PHYS2A Ratings Over Time")
plt.legend()
plt.xticks([])
plt.tight_layout()
plt.show()

In [ ]:
df_smoothed = df_course.copy()
df_smoothed["quality_avg"] = df_smoothed["quality"].ewm(span=15).mean()
df_smoothed["difficulty_avg"] = df_smoothed["difficulty"].ewm(span=15).mean()

plt.figure(figsize=(10,5))
plt.plot(df_smoothed["date"], df_smoothed["quality_avg"], linewidth=3, label="quality")
plt.plot(df_smoothed["date"], df_smoothed["difficulty_avg"], linewidth=3, label="difficulty")
plt.ylim(1,5)
plt.xlabel("Date")
plt.ylabel("Rating Score")
plt.title("PHYS2A Ratings Over Time (Smoothed)")
plt.legend()
plt.xticks([])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates

releases_df = pd.read_csv(root_dir / "data" / "chatgpt_model_updates.csv")
releases_df["Time"] = pd.to_datetime(releases_df["Time"], format="mixed")
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(df_smoothed["date"], df_smoothed["quality_avg"], linewidth=3, label="quality")
ax.plot(df_smoothed["date"], df_smoothed["difficulty_avg"], linewidth=3, label="difficulty")
ax.set_ylim(1,5)
ax.set_xlabel("Date")
ax.set_ylabel("Rating Score")
ax.set_title("PHYS2A Ratings Over Time (Smoothed)")
ax.legend()

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0, ha="center", fontsize=9)

plt.tight_layout()
for _, row in releases_df.iterrows():
    ax.axvline(row["Time"], linestyle="--", linewidth=1.2, alpha=0.6, color="black")
y_top = ax.get_ylim()[1]
for _, row in releases_df.iterrows():
    ax.text(row["Time"], y_top*0.98, row["Model"],
            rotation=90, va="top", ha="center", fontsize=10, alpha=0.8,
            color="red", fontweight="bold")
plt.show()